In [1]:
#!pip install vosk djitellopy pyaudio

In [1]:
import socket
import threading
import time
import pyaudio
from vosk import Model, KaldiRecognizer
import json
from djitellopy import Tello

# IP and port of Tello
tello_address = ('192.168.10.1', 8889)
# IP and port of local computer
local_address = ('', 9000)

# Create a UDP connection that we'll send the command to
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
# Bind to the local address and port
sock.bind(local_address)

# Send the message to Tello and allow for a delay in seconds
def send(message, delay):
    try:
        sock.sendto(message.encode(), tello_address)
        print("Sending message: " + message)
    except Exception as e:
        print("Error sending: " + str(e))
    time.sleep(delay)

# Receive the message from Tello
def receive():
    while True:
        try:
            response, ip_address = sock.recvfrom(128)
            print("Received message: " + response.decode(encoding='utf-8'))
        except Exception as e:
            sock.close()
            print("Error receiving: " + str(e))
            break

# Create and start a listening thread that runs in the background
receiveThread = threading.Thread(target=receive)
receiveThread.daemon = True
receiveThread.start()

# Put Tello into command mode
send("command", 3)

# Initialize Vosk model
model = Model(r"C:\Users\00097159\Squadrone\vosk-model-small-en-us-0.15")
recognizer = KaldiRecognizer(model, 16000)

# Initialize PyAudio
mic = pyaudio.PyAudio()
listening = False

def get_command():
    global listening
    listening = True
    stream = mic.open(format=pyaudio.paInt16, channels=1, rate=16000, input=True, frames_per_buffer=8192)
    stream.start_stream()
    while listening:
        try:
            data = stream.read(4096)
            if recognizer.AcceptWaveform(data):
                result = recognizer.Result()
                response = json.loads(result)["text"]
                print(response)
                listening = False
                stream.stop_stream()
                stream.close()
                return response
        except OSError:
            pass

def analyze_command(command):
    try:
        if "take off" in command:
        # if command == "take off":
            send("takeoff", 2)
        elif "land" in command:
            send("land", 4)
        elif "move up" in command:
            send("up 100", 4)
        elif "back flip" in command:
            send("flip b", 4)
        elif "front flip" in command:
            send("flip f", 4)
        elif "say hi" in command:
            send("cw 360", 4)
        else: 
            print("I don't understand the command, add it!")
    except Exception as e:
        print(f"Error executing command: {e}")

# Main loop
while True:
    print("Waiting for command...")
    command = get_command()
    if command:
        print(f"Command received: {command}")
        analyze_command(command)
    else:
        print("No command received.")

# Clean up
sock.close()
mic.terminate()

Sending message: command
Waiting for command...
take off
Command received: take off
Sending message: takeoff
Waiting for command...
take off
Command received: take off
Sending message: takeoff
Waiting for command...
gonna take off
Command received: gonna take off
Sending message: takeoff
Waiting for command...
take off
Command received: take off
Sending message: takeoff
Waiting for command...
take off
Command received: take off
Sending message: takeoff
Waiting for command...


KeyboardInterrupt: 

In [1]:
sock.close()

NameError: name 'sock' is not defined